<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260409_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2차 해커톤 전날 메모

## 프로젝트 개요

1차 FarmTrade 시뮬레이터에서 "농산물 선물 거래 체험"을 만들었다면,
2차는 그 거래 판단의 근거가 되는 **뉴스 정보**를 제공하는 서비스다.

실제 농산물 선물 시장에서 트레이더들은 뉴스를 보고 매수/매도를 결정한다.
우크라이나 전쟁 → 밀 가격 급등, 미국 가뭄 → 옥수수 가격 상승 같은 식으로.
즉 1차(거래 체험)와 2차(뉴스 구독)는 같은 도메인 안에서 앞뒤가 맞는 서비스다.

사용자는 회원가입 후 로그인하면 관심 있는 농산물 카테고리를 구독할 수 있고,
해당 카테고리의 뉴스만 골라서 볼 수 있다.
JWT 인증으로 로그인한 유저만 접근 가능하고, 구독 정보는 MariaDB에 저장된다.

---

## 프로젝트 기본 정보

- **서비스명:** FarmTrade News (가칭)
- **주제:** 카테고리별 농산물 뉴스 구독 서비스
- **1차 연결:** 옥수수/밀/생우/대두 카테고리 그대로 사용
- **배포:** GCP VM + Cloudflare

---

## 핵심 흐름

```
회원가입 → 로그인 → JWT 발급
→ 카테고리 구독 설정 (옥수수/밀/생우/대두)
→ 구독 카테고리 기준으로 뉴스 목록 호출
→ DB에서 해당 카테고리 뉴스 SELECT
→ React 화면에 카드 형태로 렌더링
```

---

## 파일 구조

```
프로젝트/
├── server.js
├── package.json
├── .env
├── frontend/
│   ├── src/
│   │   ├── App.jsx
│   │   ├── pages/
│   │   │   ├── Login.jsx
│   │   │   ├── Register.jsx
│   │   │   └── NewsFeed.jsx
│   │   └── components/
│   │       ├── CategoryFilter.jsx
│   │       └── NewsCard.jsx
└── README.md
```

---

## DB 테이블

```sql
users         (id, username, password, created_at)
news          (id, title, content, category, created_at)
subscriptions (id, user_id, category)
```

카테고리값: `corn` / `wheat` / `cattle` / `soybean`

---

## API 목록

| method | 경로 | 역할 |
|---|---|---|
| POST | /register | 회원가입 |
| POST | /login | JWT 발급 |
| GET | /api/news/:category | 카테고리별 뉴스 조회 |
| POST | /api/news | 뉴스 등록 (시간 남으면) |
| GET | /api/subscriptions | 내 구독 카테고리 조회 |
| POST | /api/subscriptions | 구독 카테고리 저장 |

---

## 내일 실행 순서

```
1. GCP VM 켜기 + SSH 접속
2. MariaDB 설치 → DB/유저/테이블 생성
3. npm init + 패키지 설치
4. cloudflared 설치
5. server.js 작성
6. React 프론트 작성
7. 서버 실행 → 터널 연결
8. README.md 작성
```

---

## 설치 패키지

```bash
npm install express mariadb jsonwebtoken bcrypt cors dotenv
```



---



# 1. GCP VM 세팅 순서 (내일 아침용 메모)

1. VM 인스턴스 시작
2. SSH 접속
3. Node.js 설치
4. MariaDB 설치
5. cloudflared 설치
6. 방화벽 포트 확인 (3000번)

# 2. server.js

In [ ]:
const express = require('express');
const mariadb = require('mariadb');
const jwt = require('jsonwebtoken');
const bcrypt = require('bcrypt');
const cors = require('cors');
require('dotenv').config();

const app = express();
app.use(cors());
app.use(express.json());
app.use(express.static('public')); // public 폴더 정적 파일 서빙

// DB 연결 풀 생성
// 풀(pool): 연결을 미리 여러 개 만들어두고 재사용 -> 매번 새로 연결하는 것보다 빠름
const pool = mariadb.createPool({
  host: 'localhost',
  user: process.env.DB_USER,
  password: process.env.DB_PASSWORD,
  database: process.env.DB_NAME,
  connectionLimit: 5
});

// JWT 인증 미들웨어
// 미들웨어: 요청이 라우트에 도달하기 전에 먼저 실행되는 함수
const authenticate = (req, res, next) => {
  const token = req.headers['authorization']?.split(' ')[1];
  if (!token) return res.status(401).json({ error: '토큰 없음' });
  try {
    req.user = jwt.verify(token, process.env.JWT_SECRET);
    next(); // 인증 통과 -> 다음 단계로 넘어감
  } catch {
    res.status(401).json({ error: '토큰 만료 또는 유효하지 않음' });
  }
};

// 회원가입
app.post('/register', async (req, res) => {
  const { username, password } = req.body;
  const hashed = await bcrypt.hash(password, 10); // 비밀번호 암호화
  const conn = await pool.getConnection();
  try {
    await conn.query(
      'INSERT INTO users (username, password) VALUES (?, ?)',
      [username, hashed]
    );
    res.json({ message: '회원가입 완료' });
  } finally {
    conn.release(); // 연결 반납 -> 풀에 돌려줘야 다음 요청이 쓸 수 있음
  }
});

// 로그인
app.post('/login', async (req, res) => {
  const { username, password } = req.body;
  const conn = await pool.getConnection();
  try {
    const rows = await conn.query(
      'SELECT * FROM users WHERE username = ?',
      [username]
    );
    if (!rows.length) return res.status(401).json({ error: '유저 없음' });
    const valid = await bcrypt.compare(password, rows[0].password);
    if (!valid) return res.status(401).json({ error: '비밀번호 틀림' });
    const token = jwt.sign(
      { id: rows[0].id, username },
      process.env.JWT_SECRET,
      { expiresIn: '2h' }
    );
    res.json({ token });
  } finally {
    conn.release();
  }
});

// 카테고리별 뉴스 조회 (로그인 필요)
app.get('/api/news/:category', authenticate, async (req, res) => {
  const { category } = req.params; // URL에서 카테고리 꺼냄
  const conn = await pool.getConnection();
  try {
    const rows = await conn.query(
      'SELECT * FROM news WHERE category = ? ORDER BY created_at DESC',
      [category]
    );
    res.json(rows);
  } finally {
    conn.release();
  }
});

// 내 구독 카테고리 조회
app.get('/api/subscriptions', authenticate, async (req, res) => {
  const conn = await pool.getConnection();
  try {
    const rows = await conn.query(
      'SELECT category FROM subscriptions WHERE user_id = ?',
      [req.user.id]
    );
    res.json(rows);
  } finally {
    conn.release();
  }
});

// 구독 카테고리 저장
app.post('/api/subscriptions', authenticate, async (req, res) => {
  const { category } = req.body;
  const conn = await pool.getConnection();
  try {
    await conn.query(
      'INSERT IGNORE INTO subscriptions (user_id, category) VALUES (?, ?)',
      [req.user.id, category] // INSERT IGNORE: 이미 있으면 무시
    );
    res.json({ message: '구독 완료' });
  } finally {
    conn.release();
  }
});

app.listen(3000, () => console.log('서버 실행 중 - 포트 3000'));

## .env

DB_USER=farmuser

DB_PASSWORD=비밀번호

DB_NAME=farmdb

JWT_SECRET=아무문자열이나

## 설치 패키지

bashnpm install express mariadb jsonwebtoken bcrypt cors dotenv

# 3. 파일 구조 + index.html + app.js

## 파일 구조


```
프로젝트/
├── server.js
├── package.json
├── .env
└── public/
    ├── index.html
    └── app.js
```


# index.html

In [ ]:
<!DOCTYPE html>
<html lang="ko">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>FarmTrade News</title>
  <script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-gray-50 min-h-screen">

  <!-- 로그인 화면 -->
  <div id="loginPage" class="flex flex-col items-center justify-center min-h-screen">
    <h1 class="text-2xl font-bold mb-6">🌾 FarmTrade News</h1>
    <input id="loginUsername" type="text" placeholder="아이디"
      class="border p-2 mb-3 w-64 rounded" />
    <input id="loginPassword" type="password" placeholder="비밀번호"
      class="border p-2 mb-3 w-64 rounded" />
    <button onclick="login()"
      class="bg-green-600 text-white px-6 py-2 w-64 rounded mb-2">로그인</button>
    <button onclick="showRegister()"
      class="text-sm text-gray-500">회원가입</button>
  </div>

  <!-- 회원가입 화면 -->
  <div id="registerPage" class="hidden flex flex-col items-center justify-center min-h-screen">
    <h1 class="text-2xl font-bold mb-6">회원가입</h1>
    <input id="regUsername" type="text" placeholder="아이디"
      class="border p-2 mb-3 w-64 rounded" />
    <input id="regPassword" type="password" placeholder="비밀번호"
      class="border p-2 mb-3 w-64 rounded" />
    <button onclick="register()"
      class="bg-green-600 text-white px-6 py-2 w-64 rounded mb-2">가입하기</button>
    <button onclick="showLogin()"
      class="text-sm text-gray-500">로그인으로 돌아가기</button>
  </div>

  <!-- 뉴스피드 화면 -->
  <div id="feedPage" class="hidden max-w-2xl mx-auto p-4">
    <div class="flex justify-between items-center mb-4">
      <h1 class="text-xl font-bold">🌾 FarmTrade News</h1>
      <button onclick="logout()" class="text-sm text-gray-500">로그아웃</button>
    </div>
    <!-- 카테고리 탭 -->
    <div class="flex gap-2 mb-4">
      <button onclick="selectCategory('corn')" id="tab-corn"
        class="tab px-4 py-2 rounded-full text-sm font-medium bg-gray-100">🌽 옥수수</button>
      <button onclick="selectCategory('wheat')" id="tab-wheat"
        class="tab px-4 py-2 rounded-full text-sm font-medium bg-gray-100">🌾 밀</button>
      <button onclick="selectCategory('cattle')" id="tab-cattle"
        class="tab px-4 py-2 rounded-full text-sm font-medium bg-gray-100">🐄 생우</button>
      <button onclick="selectCategory('soybean')" id="tab-soybean"
        class="tab px-4 py-2 rounded-full text-sm font-medium bg-gray-100">🫘 대두</button>
    </div>
    <!-- 뉴스 목록 -->
    <div id="newsList"></div>
  </div>

  <script src="app.js"></script>
</body>
</html>

# app.js

In [ ]:
let token = localStorage.getItem('token');

// 앱 시작 시 토큰 있으면 피드로 바로 이동
if (token) showFeed();

// --- 화면 전환 ---
function showLogin() {
  document.getElementById('loginPage').classList.remove('hidden');
  document.getElementById('registerPage').classList.add('hidden');
  document.getElementById('feedPage').classList.add('hidden');
}

function showRegister() {
  document.getElementById('loginPage').classList.add('hidden');
  document.getElementById('registerPage').classList.remove('hidden');
}

function showFeed() {
  document.getElementById('loginPage').classList.add('hidden');
  document.getElementById('registerPage').classList.add('hidden');
  document.getElementById('feedPage').classList.remove('hidden');
  selectCategory('corn'); // 기본 카테고리 옥수수
}

// --- 회원가입 ---
async function register() {
  const username = document.getElementById('regUsername').value;
  const password = document.getElementById('regPassword').value;
  const res = await fetch('/register', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({ username, password })
  });
  const data = await res.json();
  alert(data.message || data.error);
  if (data.message) showLogin();
}

// --- 로그인 ---
async function login() {
  const username = document.getElementById('loginUsername').value;
  const password = document.getElementById('loginPassword').value;
  const res = await fetch('/login', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify({ username, password })
  });
  const data = await res.json();
  if (data.token) {
    token = data.token;
    localStorage.setItem('token', token); // 토큰 저장
    showFeed();
  } else {
    alert(data.error);
  }
}

// --- 로그아웃 ---
function logout() {
  localStorage.removeItem('token');
  token = null;
  showLogin();
}

// --- 카테고리 선택 + 뉴스 불러오기 ---
async function selectCategory(category) {
  // 탭 스타일 업데이트
  document.querySelectorAll('.tab').forEach(btn => {
    btn.classList.remove('bg-green-600', 'text-white');
    btn.classList.add('bg-gray-100', 'text-gray-600');
  });
  const activeTab = document.getElementById(`tab-${category}`);
  activeTab.classList.add('bg-green-600', 'text-white');
  activeTab.classList.remove('bg-gray-100', 'text-gray-600');

  // 뉴스 fetch
  const res = await fetch(`/api/news/${category}`, {
    headers: { Authorization: `Bearer ${token}` }
  });
  const news = await res.json();
  renderNews(news);
}

// --- 뉴스 렌더링 ---
function renderNews(newsList) {
  const container = document.getElementById('newsList');
  if (!newsList.length) {
    container.innerHTML = '<p class="text-gray-400 text-sm">등록된 뉴스가 없습니다.</p>';
    return;
  }
  container.innerHTML = newsList.map(item => `
    <div class="border rounded-lg p-4 mb-3 bg-white">
      <p class="text-xs text-gray-400 mb-1">
        ${new Date(item.created_at).toLocaleDateString('ko-KR')}
      </p>
      <h2 class="font-semibold mb-2">${item.title}</h2>
      <p class="text-sm text-gray-600">${item.content}</p>
    </div>
  `).join('');
}

# DB 테이블 생성 SQL

In [ ]:
CREATE DATABASE IF NOT EXISTS farmdb;
USE farmdb;

CREATE TABLE IF NOT EXISTS users (
  id INT AUTO_INCREMENT PRIMARY KEY,
  username VARCHAR(50) UNIQUE NOT NULL,
  password VARCHAR(255) NOT NULL,
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS news (
  id INT AUTO_INCREMENT PRIMARY KEY,
  title VARCHAR(255) NOT NULL,
  content TEXT NOT NULL,
  category VARCHAR(20) NOT NULL,
  created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE IF NOT EXISTS subscriptions (
  id INT AUTO_INCREMENT PRIMARY KEY,
  user_id INT NOT NULL,
  category VARCHAR(20) NOT NULL,
  UNIQUE KEY unique_sub (user_id, category)
);

-- 테스트용 뉴스 데이터
INSERT IGNORE INTO news (title, content, category) VALUES
('옥수수 선물 가격 급등', '미국 중서부 가뭄으로 옥수수 생산량이 감소하며 선물 가격이 상승했다.', 'corn'),
('밀 수출 제한 조치', '우크라이나 정부가 밀 수출 쿼터를 축소하며 글로벌 밀 가격에 영향을 미치고 있다.', 'wheat'),
('생우 선물 시장 동향', '미국 육우 재고 감소로 생우 선물 가격이 강세를 보이고 있다.', 'cattle'),
('대두 중국 수요 증가', '중국의 대두 수입량이 전년 대비 15% 증가하며 대두 선물 가격이 상승했다.', 'soybean');



---



# 🌾 FarmTrade News

## 1. 프로젝트 개요

| 항목 | 내용 |
|---|---|
| 수행 주제 | 농산물 카테고리별 뉴스 구독 서비스 |
| 배포 주소 | https://배포후입력 |
| 사용 기술 | HTML, CSS, Tailwind CSS, Vanilla JS, Node.js, MariaDB, JWT, GCP, Cloudflare |

1차 해커톤에서 농산물 선물 거래 시뮬레이터(FarmTrade)를 만들었다면,
2차에서는 그 거래 판단의 근거가 되는 뉴스 정보를 제공하는 서비스를 구현했다.
실제 트레이더들이 뉴스를 보고 매수/매도를 결정하는 것처럼,
옥수수/밀/생우/대두 카테고리별 뉴스를 구독하고 확인할 수 있다.

---

## 2. 백엔드 구성 및 라우팅

server.js에서 설정한 주요 API 경로와 역할

| Method | 경로 | 역할 |
|---|---|---|
| POST | /register | 회원가입 (bcrypt 비밀번호 암호화) |
| POST | /login | 로그인 + JWT 토큰 발급 |
| GET | /api/news/:category | 카테고리별 뉴스 조회 (JWT 인증 필요) |
| GET | /api/subscriptions | 내 구독 카테고리 목록 조회 |
| POST | /api/subscriptions | 구독 카테고리 저장 |

인증이 필요한 API는 `authenticate` 미들웨어를 통해
요청 헤더의 JWT 토큰을 검증한 후에만 데이터를 반환한다.

---

## 3. 데이터베이스 및 SQL 활용

**사용 테이블**

| 테이블 | 역할 |
|---|---|
| users | 회원 정보 저장 (id, username, password) |
| news | 뉴스 데이터 저장 (id, title, content, category) |
| subscriptions | 유저별 구독 카테고리 저장 (user_id, category) |

**주요 SQL**

```sql
-- 카테고리별 뉴스 최신순 조회
SELECT * FROM news WHERE category = ? ORDER BY created_at DESC;

-- 로그인한 유저의 구독 카테고리만 조회
SELECT category FROM subscriptions WHERE user_id = ?;

-- 중복 구독 방지 (이미 있으면 무시)
INSERT IGNORE INTO subscriptions (user_id, category) VALUES (?, ?);
```

---

## 4. 인프라 및 배포 기록

**클라우드 서버**
- GCP VM 인스턴스 생성 후 Node.js, MariaDB 설치
- `/api/news`, `/login` 등 API 서버를 포트 3000에서 실행
- `express.static('public')`으로 프론트엔드 정적 파일 함께 서빙

**도메인 연결**
- cloudflared 설치 후 터널 생성
- Cloudflare를 통해 HTTPS 보안 접속 설정
- 외부에서 도메인으로 접속 시 GCP VM 포트 3000으로 연결

---

## 5. 트러블슈팅

| 문제 | 원인 | 해결 |
|---|---|---|
| (작성 예시) API 호출 시 401 에러 | Authorization 헤더 누락 | fetch 요청에 `Bearer ${token}` 헤더 추가 |
| (작성 예시) DB 연결 실패 | .env 파일 경로 오류 | dotenv 설정 위치 server.js 최상단으로 이동 |

> 실제 개발 중 발생한 에러로 교체할 것

---

## 6. 최종 회고

**배운 점**
- JWT 인증 흐름: 토큰 발급 → 헤더 전송 → 미들웨어 검증의 전체 사이클을 직접 구현했다
- MariaDB 연결 풀(pool) 개념과 `conn.release()`로 연결을 반납하는 방식을 이해했다
- 카테고리별 동적 SQL 쿼리를 라우트 파라미터(`req.params`)와 연결하는 구조를 익혔다

**개선 계획**
- 뉴스 직접 등록 기능 (관리자용 POST /api/news)
- 실제 농산물 뉴스 API 연동 (현재는 샘플 데이터)
- 1차 FarmTrade 시뮬레이터와 연동하여 뉴스 → 거래 판단 흐름 완성